In [2]:
import requests
import json
from httpx import HTTPError

In [3]:

def _process_response(r):
    try:
        r.raise_for_status()
        return r.json()
    except (json.JSONDecodeError, HTTPError) as e:
        if isinstance(e, json.JSONDecodeError):
            error = str(e)
        else:
            status_code = e.status_code
            reason = e.reason
            details = e.details  # Includes additional info like headers
            error = f"{reason}: {details}"
        raise Exception(f"Status: {e.status_code}, Error: {error}")
    except Exception as e:
        raise Exception(str(e))
# #| export
# def _process_response(r):
#     "Process response: raise for status and return json if possible"
#     try:
#         r = r.json()
#     except Exception as e:
#         raise Exception(f'failed to decode response: {e}, {r.text}')
#     try:
#         r.raise_for_status()
#     except Exception as e:
#         raise Exception(f'{r.status_code}: {r}')
#     return r


In [10]:
requests.Response??

Init signature: requests.Response()
Source:        
class Response:
    """The :class:`Response <Response>` object, which contains a
    server's response to an HTTP request.
    """

    __attrs__ = [
        "_content",
        "status_code",
        "headers",
        "url",
        "history",
        "encoding",
        "reason",
        "cookies",
        "elapsed",
        "request",
    ]

    def __init__(self):
        self._content = False
        self._content_consumed = False
        self._next = None

        #: Integer Code of responded HTTP Status, e.g. 404 or 200.
        self.status_code = None

        #: Case-insensitive Dictionary of Response Headers.
        #: For example, ``headers['content-encoding']`` will return the
        #: value of a ``'Content-Encoding'`` response header.
        self.headers = CaseInsensitiveDict()

        #: File-like object representation of response (for advanced usage).
        #: Use of ``raw`` requires that ``stream=True`` be set 

In [12]:
r = requests.Response()
r.status_code = 422
r.json_content = {'error': 'Field is missing'}


In [13]:
_process_response(r)

Exception: 422 Client Error: None for url: None

In [ ]:

try:
    result = _process_response(response)
    assert isinstance(result, dict), "Expected a dictionary response"
    assert result['status'] == 'decoding_error', "Expected decoding error status"
    assert result['error'].startswith("HTTP 422"), "Expected HTTP 422 error message"
except Exception as e:
    print(f"Error: {e}")

In [ ]:
```

**Cell 2: Test 400 (Bad Request)**
```python
response = requests.Response()
response.status_code = 400
response.json_content = {'error': 'Invalid request'}

try:
    result = _process_response(response)
    assert isinstance(result, dict), "Expected a dictionary response"
    assert result['status'] == f'HTTP {response.status_code}', "Expected HTTP error status"
    assert result['error'].startswith(f"HTTP {response.status_code}"), "Expected HTTP error message"
except Exception as e:
    print(f"Error: {e}")
```

**Cell 3: Test Parsing Error (200 OK but with parsing error)**
```python
response = requests.Response()
response.status_code = 200
response.json_content = None

try:
    result = _process_response(response)
    assert isinstance(result, dict), "Expected a dictionary response"
    assert result['status'] == 'decoding_error', "Expected decoding error status"
    assert result['error'].startswith("JSON Decode Error"), "Expected JSON decode error message"
except Exception as e:
    print(f"Error: {e}")
```

**Cell 4: Test 500 (Internal Server Error)**
```python
response = requests.Response()
response.status_code = 500

try:
    result = _process_response(response)
    assert isinstance(result, dict), "Expected a dictionary response"
    assert result['status'] == f'HTTP {response.status_code}', "Expected HTTP error status"
    assert result['error'].startswith(f"HTTP {response.status_code}"), "Expected HTTP error message"
except Exception as e:
    print(f"Error: {e}")